In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

In [2]:
#Load dataset
housing = fetch_california_housing()

#Convert to DataFrame
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['MedHouseVal'] = housing.target   # target = median house value

print(df.head())
print(df.shape)
print(df.info())
print(df.describe())

   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  MedHouseVal  
0    -122.23        4.526  
1    -122.22        3.585  
2    -122.24        3.521  
3    -122.25        3.413  
4    -122.25        3.422  
(20640, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   MedInc       20640 non-null  float64
 1   HouseAge     20640 non-null  float64
 2   AveRooms     20640 non-null  float64
 3   AveBedrms    20640 non-null  float64
 4  

In [3]:
#Check missing values
print(df.isnull().sum())

MedInc         0
HouseAge       0
AveRooms       0
AveBedrms      0
Population     0
AveOccup       0
Latitude       0
Longitude      0
MedHouseVal    0
dtype: int64


In [4]:
df = df.fillna(df.median())

In [5]:
#Split features and target
X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [6]:
#Feature scaling (Standardization)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

StandardScaler: features have different scales (income, rooms, latitude, etc.). Needed especially for Linear Regression and SVR. Tree models don’t require scaling, but scaling everything keeps the pipeline consistent

Regression Algorithm Implementation

In [7]:
#Helper function
def evaluate_model(model, X_train, X_test, y_train, y_test, name):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f"\n{name}")
    print("MSE :", mse)
    print("MAE :", mae)
    print("R2  :", r2)

    return {"Model": name, "MSE": mse, "MAE": mae, "R2": r2}

In [8]:
#Linear Regression
results = []
results.append(evaluate_model(
    LinearRegression(),
    X_train_scaled, X_test_scaled, y_train, y_test,
    "Linear Regression"
))


Linear Regression
MSE : 0.555891598695244
MAE : 0.5332001304956565
R2  : 0.5757877060324511


Finds a straight line relationship between features and house price. Simple and fast, good as a baseline when relationships are roughly linear.

In [9]:
#Decision Tree Regressor - Splits data into regions using feature thresholds. Captures non-linear patterns - can overfit if the tree is too deep.
results.append(evaluate_model(
    DecisionTreeRegressor(random_state=42),
    X_train_scaled, X_test_scaled, y_train, y_test,
    "Decision Tree Regressor"
))


Decision Tree Regressor
MSE : 0.49396854311945243
MAE : 0.45390448401162786
R2  : 0.6230424613065773


In [10]:
#Random Forest Regressor - Many decision trees combined (bagging). Reduces overfitting vs one tree. usually strong on tabular housing data.
results.append(evaluate_model(
    RandomForestRegressor(n_estimators=100, random_state=42),
    X_train_scaled, X_test_scaled, y_train, y_test,
    "Random Forest Regressor"
))


Random Forest Regressor
MSE : 0.255169737347244
MAE : 0.3274252027374032
R2  : 0.8052747336256919


In [11]:
#Gradient Boosting Regressor - Builds trees one after another, each fixing previous errors. Often very accurate for structured data like California Housing.
results.append(evaluate_model(
    GradientBoostingRegressor(random_state=42),
    X_train_scaled, X_test_scaled, y_train, y_test,
    "Gradient Boosting Regressor"
))


Gradient Boosting Regressor
MSE : 0.29399901242474274
MAE : 0.37165044848436773
R2  : 0.7756433164710084


In [ ]:
#Support Vector Regressor (SVR) - Fits a model within a margin of tolerance. Works well with scaled features and non-linear kernels, can be slow on large datasets.

# X_tr, y_tr = X_train_scaled[:5000], y_train.iloc[:5000]

results.append(evaluate_model(
    SVR(kernel='rbf'),
    X_train_scaled, X_test_scaled, y_train, y_test,
    "Support Vector Regressor"
))

Model Evaluation and Comparison

In [ ]:
#Compare all results
results_df = pd.DataFrame(results)
print(results_df.sort_values(by='R2', ascending=False))

In [ ]:
#Optional bar chart
results_df.plot(x='Model', y=['MSE', 'MAE', 'R2'], kind='bar', figsize=(10, 5))
plt.title('Model Comparison')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Best: Random Forest Regressor — highest R2 (0.805) and lowest MSE/MAE.
It handles non-linear patterns well and is more stable than a single Decision Tree.

Worst: Linear Regression — lowest R2 (0.576) and highest error.
It assumes a linear relationship, which is too simple for this housing data.